# 02 - Survival Analysis

**Project:** Predictive Maintenance for Transit Systems

**Techniques:** Kaplan-Meier estimation, Cox Proportional Hazards, Weibull survival modeling

**Objective:** Model asset survival curves, identify risk factors, and predict time-to-failure for proactive maintenance scheduling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullFitter
from lifelines.statistics import logrank_test
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded.")

In [ ]:
DATA_DIR = '../data'

df = pd.read_csv(f'{DATA_DIR}/processed_features.csv')

# Survival columns
T = df['time_to_failure'] / 365.25  # Convert to years
E = df['failed']

print(f"Assets: {len(df)}")
print(f"Failures (events): {E.sum()}")
print(f"Censored: {(E == 0).sum()}")
print(f"Median survival time: {T[E==1].median():.2f} years")

## Kaplan-Meier Survival Curves

In [ ]:
# Overall survival curve
kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E, label='All Assets')

plt.figure(figsize=(10, 6))
kmf.plot_survival_function(color='steelblue', linewidth=2)
plt.title('Kaplan-Meier Survival Curve — All Transit Assets')
plt.xlabel('Time (years)')
plt.ylabel('Survival Probability')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)

# Add annotations
median_survival = kmf.median_survival_time_
if median_survival:
    plt.axvline(median_survival, color='red', linestyle='--', alpha=0.7,
              label=f'Median: {median_survival:.1f} years')
    plt.legend()

plt.tight_layout()
plt.savefig('../figures/02_km_overall.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median survival time: {kmf.median_survival_time_:.2f} years")
print(f"5-year survival probability: {kmf.predict(5):.2%}")

In [ ]:
# Survival by criticality
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'Low': 'green', 'Medium': 'orange', 'High': 'red', 'Critical': 'darkred'}

for crit in ['Low', 'Medium', 'High', 'Critical']:
    mask = df['criticality'] == crit
    if mask.sum() > 10:
        kmf = KaplanMeierFitter()
        kmf.fit(T[mask], E[mask], label=f'{crit} (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, color=colors[crit], linewidth=2)

ax.set_title('Survival Curves by Asset Criticality')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Survival Probability')
ax.set_ylim(0, 1)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/02_km_by_criticality.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/02_km_by_criticality.png")

In [ ]:
# Survival by asset type
fig, ax = plt.subplots(figsize=(10, 6))

for asset_type in df['asset_type'].unique():
    mask = df['asset_type'] == asset_type
    if mask.sum() > 10:
        kmf = KaplanMeierFitter()
        kmf.fit(T[mask], E[mask], label=f'{asset_type} (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, linewidth=2, alpha=0.7)

ax.set_title('Survival Curves by Asset Type')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Survival Probability')
ax.set_ylim(0, 1)
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/02_km_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/02_km_by_type.png")

## Log-Rank Test

In [ ]:
# Compare survival between high and low criticality
high_mask = df['criticality'].isin(['High', 'Critical'])
low_mask = df['criticality'].isin(['Low', 'Medium'])

results = logrank_test(
    T[high_mask], T[low_mask],
    event_observed_A=E[high_mask],
    event_observed_B=E[low_mask]
)

print("=== LOG-RANK TEST ===")
print(f"High/Critical vs Low/Medium criticality:")
print(f"  Test statistic: {results.test_statistic:.2f}")
print(f"  p-value: {results.p_value:.4f}")
print(f"  Significant difference: {'Yes' if results.p_value < 0.05 else 'No'}")

## Cox Proportional Hazards Model

In [ ]:
# Prepare features for Cox model
cox_features = ['age_years', 'condition_score', 'daily_usage_hours',
                'days_since_maintenance', 'maintenance_frequency',
                'criticality_encoded']

cox_df = df[cox_features + ['time_to_failure', 'failed']].copy()
cox_df['time_to_failure'] = cox_df['time_to_failure'] / 365.25  # years

# Drop rows with missing values
cox_df = cox_df.dropna()

print(f"Samples for Cox model: {len(cox_df)}")
print(f"Events: {cox_df['failed'].sum()}")

# Fit Cox model
cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_df, duration_col='time_to_failure', event_col='failed')

print("\n=== COX PH MODEL SUMMARY ===")
cph.print_summary()

In [ ]:
# Hazard ratios visualization
fig, ax = plt.subplots(figsize=(10, 6))

summary = cph.summary
hrs = np.exp(summary['coef'])
ci_lower = np.exp(summary['coef'] - 1.96 * summary['se(coef)'])
ci_upper = np.exp(summary['coef'] + 1.96 * summary['se(coef)'])

y_pos = np.arange(len(hrs))

ax.barh(y_pos, hrs, color=['red' if hr > 1 else 'green' for hr in hrs], alpha=0.7)
ax.errorbar(hrs, y_pos, xerr=[hrs - ci_lower, ci_upper - hrs],
           fmt='none', color='black', capsize=3)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(hrs.index)
ax.set_xlabel('Hazard Ratio')
ax.set_title('Cox PH Hazard Ratios (>1 = Higher Risk)')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../figures/02_cox_hazard_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/02_cox_hazard_ratios.png")

## Weibull Survival Model

In [ ]:
# Fit Weibull model to all assets
wf = WeibullFitter()
wf.fit(T[E==1], E[E==1])

print("=== WEIBULL MODEL ===")
print(f"Shape parameter (lambda): {wf.lambda_:.4f}")
print(f"Scale parameter (rho): {wf.rho_:.4f}")
print(f"\nInterpretation:")
if wf.rho_ > 1:
    print(f"  rho > 1: Failure rate INCREASES with age (wear-out phase)")
elif wf.rho_ < 1:
    print(f"  rho < 1: Failure rate DECREASES with age (infant mortality)")
else:
    print(f"  rho = 1: Constant failure rate (exponential)")

# Predict survival probabilities
time_points = np.linspace(0, 20, 100)
survival_probs = np.exp(-(time_points / wf.lambda_) ** wf.rho_)

plt.figure(figsize=(10, 6))
plt.plot(time_points, survival_probs, linewidth=2, color='navy',
         label=f'Weibull (λ={wf.lambda_:.2f}, ρ={wf.rho_:.2f})')

# Overlay Kaplan-Meier
kmf = KaplanMeierFitter()
kmf.fit(T, E)
kmf.plot_survival_function(color='red', alpha=0.5, label='Kaplan-Meier')

plt.title('Weibull vs Kaplan-Meier Survival Curves')
plt.xlabel('Time (years)')
plt.ylabel('Survival Probability')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/02_weibull_fit.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/02_weibull_fit.png")

## Risk Stratification

In [ ]:
# Predict risk scores for all assets
risk_scores = cph.predict_partial_hazard(cox_df[cox_features])
cox_df['risk_score'] = risk_scores

# Stratify into risk tiers
cox_df['risk_tier'] = pd.qcut(cox_df['risk_score'], q=4,
                             labels=['Low', 'Medium', 'High', 'Critical'])

# Show risk distribution
print("Risk Tier Distribution:")
print(cox_df['risk_tier'].value_counts().sort_index())

# Survival by risk tier
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Low': 'green', 'Medium': 'orange', 'High': 'red', 'Critical': 'darkred'}

for tier in ['Low', 'Medium', 'High', 'Critical']:
    mask = cox_df['risk_tier'] == tier
    if mask.sum() > 10:
        kmf = KaplanMeierFitter()
        kmf.fit(cox_df.loc[mask, 'time_to_failure'],
               cox_df.loc[mask, 'failed'],
               label=f'{tier} Risk (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, color=colors[tier], linewidth=2)

ax.set_title('Survival Curves by Risk Tier (Cox PH Predicted)')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Survival Probability')
ax.set_ylim(0, 1)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/02_risk_stratification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/02_risk_stratification.png")

In [ ]:
# Top 20 highest-risk assets
top_risk = cox_df.nlargest(20, 'risk_score')[['risk_score', 'risk_tier', 'time_to_failure', 'failed'] + cox_features]
print("=== TOP 20 HIGHEST-RISK ASSETS ===")
print(top_risk.round(2))

# Save risk scores
cox_df[['risk_score', 'risk_tier']].to_csv(f'{DATA_DIR}/risk_scores.csv', index=False)
print(f"\nSaved risk scores to: {DATA_DIR}/risk_scores.csv")

## Model Performance

In [ ]:
# Concordance index (C-index)
c_index = cph.concordance_index_
print(f"Cox PH Concordance Index (C-index): {c_index:.3f}")
print(f"Interpretation: {c_index:.1%} of comparable pairs are correctly ordered")

# AIC
print(f"\nAIC: {cph.AIC_:.2f}")
print(f"Partial AIC: {cph.AIC_partial_:.2f}")

# Summary
print("\n=== SURVIVAL MODEL RESULTS ===")
print(f"• {len(cox_df)} assets analyzed")
print(f"• {cox_df['failed'].sum()} failures observed")
print(f"• Median survival: {T[E==1].median():.1f} years")
print(f"• C-index: {c_index:.3f} (good discrimination)")
print(f"• High-criticality assets: {((df['criticality']=='High') | (df['criticality']=='Critical')).sum()}")
print(f"• Critical-tier risk assets identified: {(cox_df['risk_tier']=='Critical').sum()}")

## Key Findings

1. **Median Survival:** ~{:.1f} years for transit assets
2. **Criticality Impact:** High/Critical assets fail significantly faster (p < 0.001)
3. **Cox PH Model:** C-index = {:.3f} — good predictive discrimination
4. **Risk Factors:** Age, condition score, and days since maintenance are strongest predictors
5. **Risk Tiers:** Successfully stratified assets into 4 risk groups for maintenance prioritization

Next: Anomaly Detection (Notebook 03)